In [ ]:
import marimo as mo
from pathlib import Path
import os

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

from PIL import Image
import torchvision.models as models
import torchvision.transforms as T

In [ ]:
WDIR = 'data/cv-lego/'
collections = ['harry-potter', 'jurassic-world', 'marvel', 'star-wars']
TDIR = Path('data/cv-lego/test')
INDEXES = Path('data/cv-lego/index.csv')

In [ ]:
for theme in collections:
    count_photo = 0
    dirs = os.walk(WDIR + theme)
    for dir in dirs:
        count_photo += len(dir[2])
    print(f'Коллекция {theme}, кол-во фото {count_photo}')
print()
print(f'Коллекция test, кол-во фото {len(os.listdir(TDIR))}')

In [ ]:
indexes = pd.read_csv(INDEXES)

In [ ]:
indexes.class_id = indexes.class_id.map(lambda cl: cl - 1)

In [ ]:
classes = indexes.class_id.nunique()
classes

In [ ]:
test_dataset = pd.read_csv('data/cv-lego/test.csv')
test_dataset.class_id = test_dataset.class_id.map(lambda id: id - 1)

In [ ]:
dataset = indexes.copy()
dataset['vectorize'] = None

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

In [ ]:
tv_model = nn.Sequential(*list(model.children())[:-1])

In [ ]:
mean_n_tv = [0.485, 0.456, 0.406]
std_n_tv  = [0.229, 0.224, 0.225]

In [ ]:
preprocessing = T.Compose([
    T.Resize(256),
    T.CenterCrop(244),
    T.ToTensor(),
    T.Normalize(
        mean=mean_n_tv,
        std=std_n_tv
    )
])

In [ ]:
def img_to_vec(path):
    image = Image.open(path).convert("RGB")
    vec = preprocessing(image).unsqueeze(0)

    with torch.no_grad():
        vector = tv_model(vec)

    vector = vector.squeeze()
    return vector.numpy()

In [ ]:
dataset.vectorize = dataset.path.map(lambda path: img_to_vec(WDIR + path))

In [ ]:
test_dataset['vectorize'] = test_dataset.path.map(lambda path: img_to_vec(WDIR + path))

In [ ]:
x_t = torch.tensor(dataset['vectorize'])
y_t = torch.tensor(dataset.class_id)

x_t_t, x_t_v, y_t_t, y_t_v = train_test_split(x_t, y_t, test_size=0.2, stratify=y_t)

In [ ]:
test_x_t = torch.tensor(test_dataset['vectorize'])
test_y_t = torch.tensor(test_dataset.class_id)

In [ ]:
tensor_dataset_train = TensorDataset(x_t_t, y_t_t)
tensor_dataset_valid = TensorDataset(x_t_v, y_t_v)
tensor_dataset_full = TensorDataset(x_t, y_t)

dataloader_train = DataLoader(tensor_dataset_train, batch_size=64, shuffle=True)
dataloader_valid = DataLoader(tensor_dataset_valid, batch_size=64, shuffle=True)
dataloader_full = DataLoader(tensor_dataset_full, batch_size=64)

In [ ]:
tensor_dataset_test = TensorDataset(test_x_t, test_y_t)
dataloader_test = DataLoader(tensor_dataset_test)

In [ ]:
class MLP(nn.Module):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.net = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
mlp = MLP()
optim = torch.optim.AdamW(mlp.parameters(), lr=0.001)

class_counts = np.bincount(y_t.numpy())
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)
loss_func = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32))

In [ ]:
def train(model: nn.Module, optimizer, loss_func, data, epoch: int = 10):
    model.train()
    loss_mean = 0
    for epo in range(epoch):
        for xb, yb in data:
            pred = model(xb)
            loss = loss_func(pred, yb)
            loss_mean += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        if epo % 10 == 0 or epo == epoch - 1:
            print(f'Эпоха: {epo}, ошибка: {np.mean(loss_mean)}')
            loss_mean = 0

In [ ]:
train(mlp, optim, loss_func, dataloader_train, 30)

In [ ]:
def model_validate(model: nn.Module, loader: torch.utils.data.DataLoader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            outputs = model(xb)
            _, predicted = torch.max(outputs.data, 1)
            total += yb.size(0)
            correct += (predicted == yb).sum().item()
        accuracy = 100 * correct / total
    print(f'\nТочность на валидации: {accuracy:.2f}')
    return accuracy

In [ ]:
model_validate(mlp, dataloader_valid)

In [ ]:
train(mlp, optim, loss_func, dataloader_full, 30)

In [ ]:
model_validate(mlp, dataloader_test)